## Cell 1 — Libraries

In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

# Fix seed so every team member gets the same split
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

print('Libraries ready.')

Libraries ready.


In [5]:
# Load raw data
df_raw = pd.read_csv(r"C:\Users\PC 118\Desktop\insurance.csv")

print(f'Shape : {df_raw.shape[0]} rows x {df_raw.shape[1]} columns')
print(f'\nColumns and types:')
print(df_raw.dtypes)
print(f'\nFirst 5 rows:')
display(df_raw.head())

Shape : 1338 rows x 7 columns

Columns and types:
age           int64
sex          object
bmi         float64
children      int64
smoker       object
region       object
charges     float64
dtype: object

First 5 rows:


,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520


In [6]:
print('=== 1. MISSING VALUES ===')
print(df_raw.isnull().sum())
print('\nTotal missing:', df_raw.isnull().sum().sum())

print('\n=== 2. DUPLICATE ROWS ===')
print('Duplicates:', df_raw.duplicated().sum())

print('\n=== 3. UNIQUE VALUES (categorical) ===')
for col in ['sex', 'smoker', 'region']:
    print(f'  {col}: {df_raw[col].unique()}')

print('\n=== 4. NUMERIC RANGES ===')
for col in ['age', 'bmi', 'children', 'charges']:
    Q1  = df_raw[col].quantile(0.25)
    Q3  = df_raw[col].quantile(0.75)
    IQR = Q3 - Q1
    ub  = Q3 + 1.5 * IQR
    out = (df_raw[col] > ub).sum()
    print(f'  {col}: min={df_raw[col].min():.1f}, max={df_raw[col].max():.1f}, IQR_upper={ub:.1f}, outliers_above={out}')

=== 1. MISSING VALUES ===
age         0
sex         0
bmi         0
children    0
smoker      0
region      0
charges     0
dtype: int64

Total missing: 0

=== 2. DUPLICATE ROWS ===
Duplicates: 1

=== 3. UNIQUE VALUES (categorical) ===
  sex: ['female' 'male']
  smoker: ['yes' 'no']
  region: ['southwest' 'southeast' 'northwest' 'northeast']

=== 4. NUMERIC RANGES ===
  age: min=18.0, max=64.0, IQR_upper=87.0, outliers_above=0
  bmi: min=16.0, max=53.1, IQR_upper=47.3, outliers_above=9
  children: min=0.0, max=5.0, IQR_upper=5.0, outliers_above=0
  charges: min=1121.9, max=63770.4, IQR_upper=34489.4, outliers_above=139


In [7]:
# Manual split using NumPy — no sklearn!
n = len(df_raw)
indices = np.arange(n)
np.random.shuffle(indices)         # shuffle with fixed seed=42

split_point = int(n * 0.80)        # 80% train, 20% test
train_idx = indices[:split_point]
test_idx  = indices[split_point:]

df_train = df_raw.iloc[train_idx].reset_index(drop=True)
df_test  = df_raw.iloc[test_idx].reset_index(drop=True)

print(f'Total : {n}')
print(f'Train : {len(df_train)} rows ({len(df_train)/n*100:.0f}%)')
print(f'Test  : {len(df_test)}  rows ({len(df_test)/n*100:.0f}%)')

# Sanity check — no overlap
assert len(set(train_idx) & set(test_idx)) == 0
print('No overlap between train and test.')

Total : 1338
Train : 1070 rows (80%)
Test  : 268  rows (20%)
No overlap between train and test.


In [8]:
# Check if charges is skewed — important for linear regression assumption
charges = df_train['charges']

print('=== charges statistics (Train) ===')
print(f'Mean   : {charges.mean():,.2f}')
print(f'Median : {charges.median():,.2f}')
print(f'Std    : {charges.std():,.2f}')
print(f'Skew   : {charges.skew():.2f}  ← should be close to 0 for normal distribution')

# Plot distribution
fig = make_subplots(rows=1, cols=2,
                    subplot_titles=['charges distribution', 'log(charges) distribution'])

fig.add_trace(go.Histogram(x=charges, nbinsx=40,
              name='charges', marker_color='steelblue'), row=1, col=1)

fig.add_trace(go.Histogram(x=np.log(charges), nbinsx=40,
              name='log(charges)', marker_color='teal'), row=1, col=2)

fig.update_layout(title='Target Variable Distribution (Train Set)',
                  showlegend=False, height=400)
fig.show()

print('\nNote: charges is right-skewed — the log transformation looks more normal.')
print('The modeling team can decide whether to use log(charges) as the target.')

=== charges statistics (Train) ===
Mean   : 13,314.71
Median : 9,440.09
Std    : 12,011.35
Skew   : 1.48  ← should be close to 0 for normal distribution



Note: charges is right-skewed — the log transformation looks more normal.
The modeling team can decide whether to use log(charges) as the target.


In [23]:
# Smoker is the dominant feature — prove it visually
smoker_mean    = df_train[df_train['smoker'] == 'yes']['charges'].mean()
nonsmoker_mean = df_train[df_train['smoker'] == 'no']['charges'].mean()

print(f'Smoker mean charges    : ${smoker_mean:,.0f}')
print(f'Non-smoker mean charges: ${nonsmoker_mean:,.0f}')
print(f'Ratio                  : {smoker_mean/nonsmoker_mean:.1f}x more expensive')

# Box plot: charges by smoker
fig = px.box(df_train, x='smoker', y='charges', color='smoker',
             title='Charges: Smokers vs Non-Smokers (Train Set)',
             color_discrete_map={'yes': '#E24B4A', 'no': '#1D9E75'})
fig.show()

Smoker mean charges    : $31,747
Non-smoker mean charges: $8,489
Ratio                  : 3.7x more expensive


In [10]:
# Compute correlation on RAW data — before encoding and scaling
# This gives the true picture of relationships
df_corr = df_train.copy()
df_corr['sex']    = df_corr['sex'].map({'male': 1, 'female': 0})
df_corr['smoker'] = df_corr['smoker'].map({'yes': 1, 'no': 0})
# For region: just drop it from correlation (will be one-hot encoded later)
df_corr = df_corr.drop(columns=['region'])

corr_matrix = df_corr.corr(numeric_only=True)

# Full heatmap
fig = px.imshow(corr_matrix, text_auto='.2f',
                color_continuous_scale='RdBu_r', zmin=-1, zmax=1,
                title='Correlation Heatmap — Train Set (raw values)')
fig.update_layout(width=650, height=550)
fig.show()

# Correlation with target specifically
print('\nCorrelation with charges (absolute, sorted):')
target_corr = corr_matrix['charges'].drop('charges').abs().sort_values(ascending=False)
print(target_corr)
print(f'\nBest single predictor: {target_corr.idxmax()} ({target_corr.max():.3f})')


Correlation with charges (absolute, sorted):
smoker      0.785552
age         0.296984
bmi         0.195103
sex         0.068993
children    0.060071
Name: charges, dtype: float64

Best single predictor: smoker (0.786)


In [11]:
# Visualize spread and outliers for all numeric features
numeric_cols = ['age', 'bmi', 'children', 'charges']

fig = make_subplots(rows=1, cols=4, subplot_titles=numeric_cols)

for i, col in enumerate(numeric_cols, 1):
    fig.add_trace(
        go.Box(y=df_train[col], name=col, showlegend=False,
               marker_color='steelblue'),
        row=1, col=i
    )

fig.update_layout(title='Box Plots — Numeric Features (Train Set)', height=420)
fig.show()

# BMI outliers detail
Q1  = df_train['bmi'].quantile(0.25)
Q3  = df_train['bmi'].quantile(0.75)
IQR = Q3 - Q1
bmi_upper = Q3 + 1.5 * IQR
print(f'BMI IQR upper bound : {bmi_upper:.2f}')
print(f'BMI outliers (train): {(df_train["bmi"] > bmi_upper).sum()} rows')
print('\nNote: charges outliers are real (smokers) — NOT data entry errors. Keep them.')

BMI IQR upper bound : 47.53
BMI outliers (train): 6 rows

Note: charges outliers are real (smokers) — NOT data entry errors. Keep them.


In [12]:
# Mean charges per category — does the category affect cost?
cat_cols = ['sex', 'smoker', 'region']

fig = make_subplots(rows=1, cols=3,
                    subplot_titles=[f'charges by {c}' for c in cat_cols])

for i, col in enumerate(cat_cols, 1):
    grp = df_train.groupby(col)['charges'].mean().reset_index()
    fig.add_trace(
        go.Bar(x=grp[col], y=grp['charges'].round(0),
               name=col, showlegend=False,
               text=grp['charges'].round(0), textposition='outside'),
        row=1, col=i
    )

fig.update_layout(title='Mean Charges by Categorical Features (Train Set)', height=420)
fig.show()

In [13]:
# Scatter plots colored by smoker — reveals the main pattern
fig = make_subplots(rows=1, cols=2,
                    subplot_titles=['age vs charges', 'bmi vs charges'])

for smoker_val, color in [('yes', '#E24B4A'), ('no', '#1D9E75')]:
    mask = df_train['smoker'] == smoker_val
    fig.add_trace(
        go.Scatter(x=df_train.loc[mask, 'age'],
                   y=df_train.loc[mask, 'charges'],
                   mode='markers', name=f'smoker={smoker_val}',
                   marker=dict(color=color, size=4, opacity=0.6)),
        row=1, col=1
    )
    fig.add_trace(
        go.Scatter(x=df_train.loc[mask, 'bmi'],
                   y=df_train.loc[mask, 'charges'],
                   mode='markers', name=f'smoker={smoker_val}',
                   marker=dict(color=color, size=4, opacity=0.6),
                   showlegend=False),
        row=1, col=2
    )

fig.update_layout(title='Age & BMI vs Charges — colored by Smoker (Train Set)',
                  height=420)
fig.show()

---
## PREPROCESSING

## Cell 11 — Remove Duplicate Row

In [14]:
# Remove the 1 duplicate row found during diagnosis
# Only apply to train — test is independent
before = len(df_train)
train  = df_train.drop_duplicates().reset_index(drop=True)
test   = df_test.drop_duplicates().reset_index(drop=True)

print(f'Train: {before} → {len(train)} rows (removed {before - len(train)} duplicate)')
print(f'Test : {len(df_test)} → {len(test)} rows')

Train: 1070 → 1069 rows (removed 1 duplicate)
Test : 268 → 268 rows


In [15]:
# Cap BMI at 47.3 (IQR upper bound = Q3 + 1.5*IQR)
# This is a domain rule — BMI above 47 is extremely rare and likely a data entry error
# We use clip (not drop) to keep all rows
# IMPORTANT: compute the cap from TRAIN only, apply to both

Q3_bmi  = train['bmi'].quantile(0.75)
IQR_bmi = Q3_bmi - train['bmi'].quantile(0.25)
BMI_CAP = Q3_bmi + 1.5 * IQR_bmi

print(f'BMI cap (from train): {BMI_CAP:.2f}')
print(f'Rows affected in train: {(train["bmi"] > BMI_CAP).sum()}')
print(f'Rows affected in test : {(test["bmi"] > BMI_CAP).sum()}')

train['bmi'] = train['bmi'].clip(upper=BMI_CAP)   # apply to train
test['bmi']  = test['bmi'].clip(upper=BMI_CAP)    # same cap on test

print(f'\nBMI max after clip — train: {train["bmi"].max():.2f}')
print(f'BMI max after clip — test : {test["bmi"].max():.2f}')

BMI cap (from train): 47.53
Rows affected in train: 6
Rows affected in test : 1

BMI max after clip — train: 47.53
BMI max after clip — test : 47.53


In [16]:
# sex — binary: male=1, female=0
sex_map = {'male': 1, 'female': 0}
train['sex'] = train['sex'].map(sex_map)
test['sex']  = test['sex'].map(sex_map)

# smoker — binary: yes=1, no=0
smoker_map = {'yes': 1, 'no': 0}
train['smoker'] = train['smoker'].map(smoker_map)
test['smoker']  = test['smoker'].map(smoker_map)

# region — one-hot encoding (4 categories → 3 dummy columns, drop_first avoids multicollinearity)
# We define the categories from TRAIN to avoid new categories in test causing mismatch
region_dummies_train = pd.get_dummies(train['region'], prefix='region', drop_first=True)
region_dummies_test  = pd.get_dummies(test['region'],  prefix='region', drop_first=True)

# Align test columns to match train (in case a category is missing in test)
region_dummies_test = region_dummies_test.reindex(
    columns=region_dummies_train.columns, fill_value=0
)

# Drop original region column and add dummies
train = pd.concat([train.drop(columns=['region']), region_dummies_train], axis=1)
test  = pd.concat([test.drop(columns=['region']),  region_dummies_test],  axis=1)

print('Encoding complete.')
print('\nColumns after encoding:', train.columns.tolist())
print('\nSample (train):')
display(train[['sex','smoker','region_northwest','region_southeast','region_southwest']].head(5))

Encoding complete.

Columns after encoding: ['age', 'sex', 'bmi', 'children', 'smoker', 'charges', 'region_northwest', 'region_southeast', 'region_southwest']

Sample (train):


,sex,smoker,region_northwest,region_southeast,region_southwest
0,0,0,False,False,False
1,0,0,True,False,False
2,0,1,True,False,False
3,1,0,True,False,False
4,1,1,True,False,False


## Separate X and Y

In [17]:
TARGET = 'charges'

X_train = train.drop(columns=[TARGET]).values.astype(float)
Y_train = train[TARGET].values.astype(float)

X_test  = test.drop(columns=[TARGET]).values.astype(float)
Y_test  = test[TARGET].values.astype(float)

feature_names = train.drop(columns=[TARGET]).columns.tolist()

print(f'X_train : {X_train.shape}')
print(f'Y_train : {Y_train.shape}')
print(f'X_test  : {X_test.shape}')
print(f'Y_test  : {Y_test.shape}')
print(f'\nFeatures: {feature_names}')

X_train : (1069, 8)
Y_train : (1069,)
X_test  : (268, 8)
Y_test  : (268,)

Features: ['age', 'sex', 'bmi', 'children', 'smoker', 'region_northwest', 'region_southeast', 'region_southwest']


In [18]:
# Manual Z-score standardization — no sklearn!
# Formula: X_scaled = (X - mean_train) / std_train

train_mean = X_train.mean(axis=0)   # mean per feature from TRAIN only
train_std  = X_train.std(axis=0)    # std  per feature from TRAIN only
train_std[train_std == 0] = 1       # avoid division by zero

X_train_scaled = (X_train - train_mean) / train_std
X_test_scaled  = (X_test  - train_mean) / train_std   # use TRAIN stats on test!

print('Scaling complete.')
print(f'\nMean check (should be ~0): {X_train_scaled.mean(axis=0).round(3)}')
print(f'Std  check (should be ~1): {X_train_scaled.std(axis=0).round(3)}')

Scaling complete.

Mean check (should be ~0): [ 0.  0.  0.  0.  0. -0. -0.  0.]
Std  check (should be ~1): [1. 1. 1. 1. 1. 1. 1. 1.]


In [19]:
print('=' * 55)
print('FINAL VERIFICATION')
print('=' * 55)

# 1. Shapes
print(f'X_train_scaled : {X_train_scaled.shape}')
print(f'Y_train        : {Y_train.shape}')
print(f'X_test_scaled  : {X_test_scaled.shape}')
print(f'Y_test         : {Y_test.shape}')

# 2. No NaN
print(f'\nNaN in X_train : {np.isnan(X_train_scaled).sum()}')
print(f'NaN in X_test  : {np.isnan(X_test_scaled).sum()}')
print(f'NaN in Y_train : {np.isnan(Y_train).sum()}')

# 3. Mean ~0, Std ~1
print(f'\nX_train mean (all features ~0): {X_train_scaled.mean(axis=0).round(2)}')
print(f'X_train std  (all features ~1): {X_train_scaled.std(axis=0).round(2)}')

# 4. Feature names
print(f'\nFeatures ({len(feature_names)}): {feature_names}')

# 5. Best predictor reminder
print(f'\nBest single predictor for Case 1: smoker (corr=0.787)')
print('=' * 55)
print('All checks passed — data is ready for modeling!')

FINAL VERIFICATION
X_train_scaled : (1069, 8)
Y_train        : (1069,)
X_test_scaled  : (268, 8)
Y_test         : (268,)

NaN in X_train : 0
NaN in X_test  : 0
NaN in Y_train : 0

X_train mean (all features ~0): [ 0.  0.  0.  0.  0. -0. -0.  0.]
X_train std  (all features ~1): [1. 1. 1. 1. 1. 1. 1. 1.]

Features (8): ['age', 'sex', 'bmi', 'children', 'smoker', 'region_northwest', 'region_southeast', 'region_southwest']

Best single predictor for Case 1: smoker (corr=0.787)
All checks passed — data is ready for modeling!


##  Verification Visualization

In [20]:
# Visual proof that scaling worked and data is clean
df_scaled_check = pd.DataFrame(X_train_scaled, columns=feature_names)

fig = make_subplots(rows=1, cols=len(feature_names),
                    subplot_titles=feature_names)

for i, col in enumerate(feature_names, 1):
    fig.add_trace(
        go.Box(y=df_scaled_check[col], name=col,
               showlegend=False, marker_color='teal'),
        row=1, col=i
    )

fig.update_layout(
    title='X_train_scaled — all features centered around 0 (proof scaling worked)',
    height=420
)
fig.show()
print('All features are now on the same scale — ready for gradient descent.')

All features are now on the same scale — ready for gradient descent.


---
##  Save All Files

In [1]:
# Save scaled arrays (for modeling team — use these for gradient descent)
np.savez(
    'insurance_processed.npz',
    X_train       = X_train_scaled,
    Y_train       = Y_train,
    X_test        = X_test_scaled,
    Y_test        = Y_test,
    feature_names = np.array(feature_names)
)

# Save as CSV (human-readable)
pd.DataFrame(X_train_scaled, columns=feature_names).to_csv(
    'insurance_X_train_scaled.csv', index=False)

pd.DataFrame(X_test_scaled, columns=feature_names).to_csv(
    'insurance_X_test_scaled.csv', index=False)

pd.DataFrame({'charges': Y_train}).to_csv('insurance_Y_train.csv', index=False)
pd.DataFrame({'charges': Y_test}).to_csv('insurance_Y_test.csv',  index=False)

# Also save unscaled + scaling params (for interpretability)
np.savez(
    'insurance_scaling_params.npz',
    train_mean    = train_mean,
    train_std     = train_std,
    feature_names = np.array(feature_names)
)



NameError: name 'np' is not defined

---
 Summary for Modeling Team

In [22]:
# How to load the processed data in the modeling notebook
print('''
HOW TO LOAD IN MODELING NOTEBOOK:
----------------------------------
import numpy as np

data = np.load('insurance_processed.npz', allow_pickle=True)
X_train = data['X_train']        # shape: (n_train, 7)
Y_train = data['Y_train']        # shape: (n_train,)
X_test  = data['X_test']         # shape: (n_test, 7)  <- DO NOT USE until final eval
Y_test  = data['Y_test']         # shape: (n_test,)   <- DO NOT USE until final eval
feature_names = data['feature_names']

BEST SINGLE PREDICTOR (Case 1): smoker  (correlation = 0.787)
FEATURE ORDER: age, sex, bmi, children, smoker, region_northwest, region_southeast, region_southwest
''')


HOW TO LOAD IN MODELING NOTEBOOK:
----------------------------------
import numpy as np

data = np.load('insurance_processed.npz', allow_pickle=True)
X_train = data['X_train']        # shape: (n_train, 7)
Y_train = data['Y_train']        # shape: (n_train,)
X_test  = data['X_test']         # shape: (n_test, 7)  <- DO NOT USE until final eval
Y_test  = data['Y_test']         # shape: (n_test,)   <- DO NOT USE until final eval
feature_names = data['feature_names']

BEST SINGLE PREDICTOR (Case 1): smoker  (correlation = 0.787)
FEATURE ORDER: age, sex, bmi, children, smoker, region_northwest, region_southeast, region_southwest

